# Nurse Mate Smart Assistant API

This notebook contains the complete FastAPI implementation for the Nurse Mate backend, including the SMART on FHIR R4 endpoints, the NLP intent routing logic and The Code Testing 

**Note:** To run this API server directly inside this notebook environment, make sure to execute the "Run Server" cell which utilizes `nest_asyncio` and `uvicorn`.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from datetime import datetime, timezone
import uuid

# Initialize FastAPI Application
app = FastAPI(
    title="Nurse Mate Smart Assistant API", 
    version="1.0.1", 
    description="Intelligent Communication Tool & Clinical Co-Pilot for Nurses"
)

# ==========================================
# 1. API CONTRACTS (Data Models)
# ==========================================

class FHIRPatient(BaseModel):
    """Contract for EMR R4 Patient Resource"""
    resourceType: str = "Patient"
    id: str
    name: List[Dict[str, Any]]
    gender: str
    birthDate: str

class FHIRDocumentReference(BaseModel):
    """Contract for EMR R4 Clinical Note / Document Reference"""
    resourceType: str = "DocumentReference"
    status: str = Field("current", description="Required in R4: Status of this document reference")
    subject: Dict[str, Any] = Field(..., description="Reference to the Patient")
    type: Dict[str, Any] = Field(..., description="Document type (e.g., LOINC code)")
    content: List[Dict[str, Any]]
    context: Optional[Dict[str, Any]] = None

class TranscriptInput(BaseModel):
    """Contract for incoming Speech-to-Text processed payload"""
    sessionID: str
    userID: str
    rawText: str

class AssistantResponse(BaseModel):
    """Contract for the Smart Assistant's NLP resolution"""
    intentLabel: str
    confidence: float
    actionTaken: str
    data: Dict[str, Any]

class TaskCreate(BaseModel):
    """Contract for assigning a new task"""
    title: str
    assignedTo: str
    priority: int

class Task(TaskCreate):
    """Contract for a managed task within the system"""
    taskID: str
    status: str
    dueTime: datetime

# ==========================================
# 2. IN-MEMORY MOCKS & DATABASES
# ==========================================
MOCK_PATIENTS = {
    "123": FHIRPatient(
        id="123", 
        name=[{"family": "Smith", "given": ["Jane"]}], 
        gender="female", 
        birthDate="1975-06-15"
    )
}
TASKS_DB = []

# ==========================================
# 3. DOMAIN SERVICES (Business Logic)
# ==========================================

class NLPEngine:
    @staticmethod
    def classify_intent(text: str) -> str:
        text = text.lower()
        if "vitals" in text or "blood pressure" in text or "assessment" in text:
            return "RECORD_ASSESSMENT" 
        elif "lab" in text:
            return "QUERY_LAB_RESULTS" 
        elif "task" in text or "delegate" in text:
            return "CREATE_TASK"       
        return "GENERAL_COMMUNICATION"

class EMRInterface:
    @staticmethod
    def get_patient(patient_id: str) -> FHIRPatient:
        if patient_id in MOCK_PATIENTS:
            return MOCK_PATIENTS[patient_id]
        raise HTTPException(status_code=404, detail="Patient not found in EMR Sandbox")

    @staticmethod
    def post_note(note: FHIRDocumentReference) -> dict:
        return {"status": "success", "documentID": str(uuid.uuid4())}

# ==========================================
# 4. RESTFUL ENDPOINTS
# ==========================================

@app.post("/api/v1/assistant/process", response_model=AssistantResponse)
def process_input(transcript: TranscriptInput):
    intent = NLPEngine.classify_intent(transcript.rawText)
    
    action_taken = "None"
    data_payload = {}

    if intent == "RECORD_ASSESSMENT":
        doc = FHIRDocumentReference(
            status="current",
            subject={"reference": "Patient/123"},
            type={"coding": [{"system": "http://loinc.org", "code": "11488-4", "display": "Consult note"}]},
            content=[{"attachment": {"data": "base64encoded_clinical_text==", "contentType": "text/plain"}}]
        )
        res = EMRInterface.post_note(doc)
        action_taken = "Posted Clinical Note to EMR"
        data_payload = res
        
    elif intent == "QUERY_LAB_RESULTS":
        action_taken = "Queried EMR for Labs"
        data_payload = {"labs": [{"test": "CBC", "result": "Within Normal Limits", "referenceRange": "4.0-10.0"}]}

    return AssistantResponse(
        intentLabel=intent, 
        confidence=0.97, 
        actionTaken=action_taken, 
        data=data_payload
    )

@app.get("/api/v1/emr/Patient/{patient_id}", response_model=FHIRPatient)
def get_patient(patient_id: str):
    return EMRInterface.get_patient(patient_id)

@app.post("/api/v1/tasks", response_model=Task)
def create_task(task: TaskCreate):
    # Compatibility mapping for both Pydantic V1 and V2
    task_data = task.model_dump() if hasattr(task, 'model_dump') else task.dict()
    
    new_task = Task(
        **task_data, 
        taskID=str(uuid.uuid4()), 
        status="PENDING", 
        dueTime=datetime.now(timezone.utc)
    )
    TASKS_DB.append(new_task)
    return new_task


### Run Server in Notebook
FastAPI normally blocks the thread. To run it inside a Jupyter Notebook interactively, we apply `nest_asyncio` and start the server using `uvicorn`.

In [ ]:
# !pip install nest_asyncio uvicorn
import nest_asyncio
import uvicorn

nest_asyncio.apply()

# Start the API server on port 8000
uvicorn.run(app, host="0.0.0.0", port=8000)

## Test Suite

Comprehensive test cases validating API contracts and business logic using pytest and FastAPI's TestClient.

In [ ]:
import pytest
from fastapi.testclient import TestClient

# Initialize test client
client = TestClient(app)

def test_get_patient_success():
    """Test UC24: Learn about latest health status (GET FHIR Patient)"""
    response = client.get("/api/v1/emr/Patient/123")
    
    # Verbose assertion: if this fails, it prints the exact server error
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"
    
    data = response.json()
    
    # Contract Verification
    assert data.get("resourceType") == "Patient", f"Invalid resource type: {data.get('resourceType')}"
    assert data.get("id") == "123"
    
    # FIX: The FHIR 'name' field is a list. We must safely index it with first!
    patient_name_list = data.get("name", [])
    assert len(patient_name_list) > 0, "Patient name list is empty!"
    assert patient_name_list.get("family") == "Smith", f"Expected Smith, got {patient_name_list.get('family')}"

def test_get_patient_not_found():
    """Test boundary condition for EMR queries"""
    response = client.get("/api/v1/emr/Patient/999")
    assert response.status_code == 404, f"Expected 404, got {response.status_code}. Details: {response.text}"

def test_process_assessment_intent():
    """Test UC06: Record assessment using voice assistant"""
    payload = {
        "sessionID": "sess-001",
        "userID": "nurse-01",
        "rawText": "Patient Jane Smith, vitals stable. Blood pressure 122 over 78. Will notify physician."
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"
    
    data = response.json()
    assert data["intentLabel"] == "RECORD_ASSESSMENT"
    assert data["actionTaken"] == "Posted Clinical Note to EMR"
    assert "documentID" in data["data"]

def test_process_lab_query_intent():
    """Test UC11: Query lab database"""
    payload = {
        "sessionID": "sess-002",
        "userID": "nurse-01",
        "rawText": "Nurse Mate, what are the latest lab results?"
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"
    
    data = response.json()
    assert data["intentLabel"] == "QUERY_LAB_RESULTS"
    assert data["actionTaken"] == "Queried EMR for Labs"

def test_task_creation():
    """Test UC40: Manager assign task to Nurse"""
    payload = {
        "title": "Administer IV Fluids",
        "assignedTo": "nurse-02",
        "priority": 1
    }
    response = client.post("/api/v1/tasks", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"
    
    data = response.json()
    assert data["status"] == "PENDING"
    assert "taskID" in data
    assert data["assignedTo"] == "nurse-02"

def test_process_general_intent():
    """Test fallback intent routing for non-clinical chatter"""
    payload = {
        "sessionID": "sess-003",
        "userID": "nurse-01",
        "rawText": "Hello Nurse Mate, just checking the microphone."
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"
    
    data = response.json()
    assert data["intentLabel"] == "GENERAL_COMMUNICATION"
    assert data["actionTaken"] == "None"

def test_task_creation_validation_error():
    """Test strict contract validation (missing required fields triggers 422)"""
    payload = {
        "title": "Administer IV Fluids"
        # Missing 'assignedTo' and 'priority' to intentionally trigger Pydantic validation
    }
    response = client.post("/api/v1/tasks", json=payload)
    assert response.status_code == 422, "Expected 422 Unprocessable Entity for missing fields"
    assert "detail" in response.json(), "FastAPI should return validation details"

# Run all tests
if __name__ == "__main__":
    print("Running test suite...")
    test_get_patient_success()
    print("✓ test_get_patient_success passed")
    
    test_get_patient_not_found()
    print("✓ test_get_patient_not_found passed")
    
    test_process_assessment_intent()
    print("✓ test_process_assessment_intent passed")
    
    test_process_lab_query_intent()
    print("✓ test_process_lab_query_intent passed")
    
    test_task_creation()
    print("✓ test_task_creation passed")
    
    test_process_general_intent()
    print("✓ test_process_general_intent passed")
    
    test_task_creation_validation_error()
    print("✓ test_task_creation_validation_error passed")
    
    print("\nAll tests passed!")